# Long-output sweep clean-run test

This notebook simulates a training sweep where the useful evidence is emitted at the tail of a long progress log.

The test intentionally leaves `NL_ARTIFACT_DIR` and `ML_ARTIFACT_DIR` unset so Notebook Lens should infer the artifact scope from the notebook path:

```text
artifacts/agent_rounds/20260615_round7/long_output_sweep/
```

Expected evidence near the end of stdout:

- `FINAL_BEST_CONFIG_JSON=...`
- `FINAL_BEST_METRICS_JSON=...`
- `FINAL_DECISION=...`


In [1]:
from __future__ import annotations

import csv
import json
import math
import random
from pathlib import Path


ARTIFACT_DIR = Path("artifacts/agent_rounds/20260615_round7/long_output_sweep")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(20260615)

configs = []
for idx, lr in enumerate([0.005, 0.01, 0.015, 0.02, 0.03, 0.04, 0.055, 0.07, 0.09, 0.12, 0.16, 0.22]):
    configs.append(
        {
            "config_id": f"cfg_{idx:02d}",
            "learning_rate": lr,
            "weight_decay": [0.0, 0.0005, 0.001, 0.002][idx % 4],
            "batch_size": [128, 256, 512][idx % 3],
            "dropout": [0.05, 0.10, 0.15, 0.20][(idx + 1) % 4],
            "scheduler": ["cosine", "linear", "one_cycle"][idx % 3],
        }
    )


def emit(line: str) -> None:
    print(line, flush=False)


def quality_for(config: dict[str, object], epoch: int) -> tuple[float, float, float, float]:
    lr = float(config["learning_rate"])
    wd = float(config["weight_decay"])
    dropout = float(config["dropout"])
    batch = int(config["batch_size"])
    ideal_lr = 0.07
    lr_penalty = abs(math.log(lr / ideal_lr)) * 0.035
    reg_bonus = 0.018 if 0.0005 <= wd <= 0.002 else -0.006
    dropout_penalty = abs(dropout - 0.15) * 0.08
    batch_bonus = {128: -0.005, 256: 0.008, 512: 0.002}[batch]
    learning_curve = 0.50 + 0.30 * (1.0 - math.exp(-epoch / 13.0))
    wiggle = math.sin(epoch * 0.45 + lr * 11.0) * 0.004
    val_f1 = learning_curve - lr_penalty + reg_bonus - dropout_penalty + batch_bonus + wiggle
    val_f1 = max(0.35, min(0.89, val_f1))
    val_loss = 0.72 - (val_f1 - 0.50) * 0.85 + wd * 8.0 + dropout * 0.03
    accuracy = val_f1 + 0.025 + math.cos(epoch / 5.0 + lr) * 0.003
    brier = 0.25 - (val_f1 - 0.50) * 0.22 + wd * 0.8
    return round(val_loss, 4), round(accuracy, 4), round(val_f1, 4), round(brier, 4)


emit("SWEEP_START run_id=20260615_round7_long_output_sweep dataset=synthetic_churn rows=48000")
emit("SPLIT train=33600 validation=7200 test=7200 target=renewal_risk positive_rate=0.314")
emit("FEATURES numeric=42 categorical=17 text_embeddings=384 calibration=platt_scaling")
emit("SEARCH_SPACE configs=12 epochs_per_config=36 objective=max(validation_f1)")

rows = []
best = None
for config_index, config in enumerate(configs, start=1):
    emit(
        "CONFIG_START "
        f"ordinal={config_index:02d}/12 config_id={config['config_id']} "
        f"lr={config['learning_rate']} wd={config['weight_decay']} "
        f"batch={config['batch_size']} dropout={config['dropout']} scheduler={config['scheduler']}"
    )
    last_metrics = None
    for epoch in range(1, 37):
        val_loss, accuracy, val_f1, brier = quality_for(config, epoch)
        train_loss = round(val_loss + 0.11 * math.exp(-epoch / 11.0) + 0.015, 4)
        grad_norm = round(2.8 / math.sqrt(epoch + 3) + float(config["learning_rate"]) * 1.7, 4)
        tokens_per_sec = int(7800 + int(config["batch_size"]) * 4 + random.randint(-95, 95))
        emit(
            "progress "
            f"config={config['config_id']} epoch={epoch:02d}/36 "
            f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
            f"val_accuracy={accuracy:.4f} val_f1={val_f1:.4f} "
            f"brier={brier:.4f} grad_norm={grad_norm:.4f} tokens_per_sec={tokens_per_sec}"
        )
        last_metrics = {
            "config_id": config["config_id"],
            "epoch": epoch,
            "validation_loss": val_loss,
            "validation_accuracy": accuracy,
            "validation_f1": val_f1,
            "validation_brier": brier,
            "learning_rate": config["learning_rate"],
            "weight_decay": config["weight_decay"],
            "batch_size": config["batch_size"],
            "dropout": config["dropout"],
            "scheduler": config["scheduler"],
        }
        rows.append(last_metrics)
    emit(
        "CONFIG_DONE "
        f"config_id={config['config_id']} final_val_loss={last_metrics['validation_loss']:.4f} "
        f"final_val_f1={last_metrics['validation_f1']:.4f} final_brier={last_metrics['validation_brier']:.4f}"
    )
    if best is None or last_metrics["validation_f1"] > best["validation_f1"]:
        best = dict(last_metrics)

assert best is not None

test_metrics = {
    "test_accuracy": round(best["validation_accuracy"] - 0.006, 4),
    "test_f1": round(best["validation_f1"] - 0.009, 4),
    "test_brier": round(best["validation_brier"] + 0.004, 4),
    "test_log_loss": round(best["validation_loss"] + 0.012, 4),
}
best_config = {
    "config_id": best["config_id"],
    "learning_rate": best["learning_rate"],
    "weight_decay": best["weight_decay"],
    "batch_size": best["batch_size"],
    "dropout": best["dropout"],
    "scheduler": best["scheduler"],
    "selected_epoch": best["epoch"],
}
final_metrics = {
    "validation_accuracy": best["validation_accuracy"],
    "validation_f1": best["validation_f1"],
    "validation_brier": best["validation_brier"],
    **test_metrics,
}
summary = {
    "run_id": "20260615_round7_long_output_sweep",
    "configs_evaluated": len(configs),
    "epochs_per_config": 36,
    "progress_rows": len(rows),
    "best_config": best_config,
    "final_metrics": final_metrics,
}

with (ARTIFACT_DIR / "sweep_training_log.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)
(ARTIFACT_DIR / "best_config.json").write_text(json.dumps(best_config, indent=2) + "\n", encoding="utf-8")
(ARTIFACT_DIR / "final_metrics.json").write_text(json.dumps(final_metrics, indent=2) + "\n", encoding="utf-8")
(ARTIFACT_DIR / "run_summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")

emit("FINAL_SUMMARY_BEGIN")
emit("FINAL_BEST_CONFIG_JSON=" + json.dumps(best_config, sort_keys=True))
emit("FINAL_BEST_METRICS_JSON=" + json.dumps(final_metrics, sort_keys=True))
emit(
    "FINAL_DECISION="
    f"promote {best_config['config_id']} with validation_f1={final_metrics['validation_f1']:.4f} "
    f"test_f1={final_metrics['test_f1']:.4f}; artifacts={ARTIFACT_DIR.as_posix()}"
)
emit("FINAL_SUMMARY_END")


SWEEP_START run_id=20260615_round7_long_output_sweep dataset=synthetic_churn rows=48000
SPLIT train=33600 validation=7200 test=7200 target=renewal_risk positive_rate=0.314
FEATURES numeric=42 categorical=17 text_embeddings=384 calibration=platt_scaling
SEARCH_SPACE configs=12 epochs_per_config=36 objective=max(validation_f1)
CONFIG_START ordinal=01/12 config_id=cfg_00 lr=0.005 wd=0.0 batch=128 dropout=0.1 scheduler=cosine
progress config=cfg_00 epoch=01/36 train_loss=0.9091 val_loss=0.7937 val_accuracy=0.4447 val_f1=0.4168 brier=0.2683 grad_norm=1.4085 tokens_per_sec=8334
progress config=cfg_00 epoch=02/36 train_loss=0.8818 val_loss=0.7751 val_accuracy=0.4664 val_f1=0.4387 brier=0.2635 grad_norm=1.2607 tokens_per_sec=8327
progress config=cfg_00 epoch=03/36 train_loss=0.8571 val_loss=0.7584 val_accuracy=0.4859 val_f1=0.4584 brier=0.2592 grad_norm=1.1516 tokens_per_sec=8296
progress config=cfg_00 epoch=04/36 train_loss=0.8350 val_loss=0.7435 val_accuracy=0.5030 val_f1=0.4759 brier=0.2553

# Clean-run excerpt finding

`run-clean` reran the notebook successfully and reported the inferred artifact scope, but its `stdout_excerpt` only included the first two progress lines from `cfg_00`.

The decisive tail evidence was present in the notebook output and artifacts, but it was not present in the `run-clean` command output:

```text
FINAL_BEST_CONFIG_JSON={"batch_size": 256, "config_id": "cfg_07", "dropout": 0.05, "learning_rate": 0.07, "scheduler": "linear", "selected_epoch": 36, "weight_decay": 0.002}
FINAL_BEST_METRICS_JSON={"test_accuracy": 0.816, "test_brier": 0.1906, "test_f1": 0.7864, "test_log_loss": 0.4984, "validation_accuracy": 0.822, "validation_brier": 0.1866, "validation_f1": 0.7954}
FINAL_DECISION=promote cfg_07 with validation_f1=0.7954 test_f1=0.7864; artifacts=artifacts/agent_rounds/20260615_round7/long_output_sweep
```
